# Echocardiogram & Iris Preprocessing

Pipeline: **download → inspect → clean (incl. mode imputation) → split → quantile transform (fit on train only) → SMOTE if imbalanced (train only) → save**.

Every step follows the GradTree paper (Section 4, "Datasets and Preprocessing") and, where the paper gives no detail, the official GradTree code (`experiments_paper_gradtree/utilities/utilities_GDT.py`), so our results are comparable with the paper. Each split is saved as a plain CSV with the target in the last column, as in the other preprocessing notebooks.

## 1. Setup

- The package below is needed for SMOTE. (For Colab)

In [1]:
%pip install -q --disable-pip-version-check imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import io
import json
import urllib.request
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score
from imblearn.over_sampling import SMOTE, RandomOverSampler

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'preprocess':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / 'dataset'          # raw files, e.g. dataset/iris/iris.data
PREPROCESS_DIR = PROJECT_ROOT / 'preprocess'    # outputs, e.g. preprocess/iris_output/

## 2. Settings

Make sure all datasets are processed the same way, as the paper did.

In [3]:
SEEDS = [42 + i for i in range(10)]  # paper: data seed = 42 + trial, 10 trials
SEED = SEEDS[0]            # the split saved to preprocess/<name>_output/ (trial 0)
SAVE_ALL_SEEDS = False     # True: also save every seed to preprocess/<name>_output/seed_<seed>/

TEST_SIZE = 0.2            # paper: 80%/20% train-test split
VAL_SIZE = 0.2             # paper: 20% of the training data as validation -> 64/16/20 overall

DROP_DUPLICATES = False    # paper keeps duplicates; True removes Iris's duplicate rows
BALANCE_THRESHOLD = 0.25   # paper: SMOTE if smallest class share <= 0.25 / (num_classes - 1)
QUANTILE_NOISE = 1e-3      # paper adds tiny noise to train data before fitting the quantile transform
QT_OUTPUT = 'normal'       # paper uses output_distribution='normal'
QT_SEED = 42               # official code seeds the noise and the quantile transform with 42 in every trial

## 3. Download the data

The original UCI `.data` files are downloaded once and kept in `dataset/<name>/`, so everyone preprocesses exactly the same raw data. If UCI is unavailable, the copy in the GradTree repository is used instead. The files have no header row, so the column names below are the ones UCI lists for each dataset.

In [4]:
UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases'
MIRROR_URL = 'https://raw.githubusercontent.com/s-marton/GradTree/master/experiments_paper_gradtree/real_world_datasets'

ECHO_COLUMNS = [
    'survival',               # months survived (or followed, if still alive); alive-at-1 is derived from it -> leakage
    'still-alive',            # 0 = dead at end of survival period, 1 = still alive; also used to derive alive-at-1 -> leakage
    'age-at-heart-attack',
    'pericardial-effusion',   # binary: fluid around the heart
    'fractional-shortening',  # contractility, lower = more abnormal
    'epss',                   # E-point septal separation, larger = more abnormal
    'lvdd',                   # left ventricular end-diastolic dimension, large hearts tend to be sick
    'wall-motion-score',
    'wall-motion-index',      # wall-motion-score / number of segments seen
    'mult',                   # derived variable
    'name',                   # always "name" -> no information
    'group',                  # "meaningless" per UCI
    'alive-at-1',             # target: 1 = alive one year after the heart attack
]
IRIS_COLUMNS = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']

# name: (download URLs, column names, target column)
SOURCES = {
    'echocardiogram': ([f'{UCI_URL}/echocardiogram/echocardiogram.data',
                        f'{MIRROR_URL}/Echocardiogram/echocardiogram.data'],
                       ECHO_COLUMNS, 'alive-at-1'),
    'iris': ([f'{UCI_URL}/iris/iris.data', f'{MIRROR_URL}/Iris/iris.data'],
             IRIS_COLUMNS, 'class'),
}

def download(urls, path):
    if path.exists() and path.stat().st_size > 0:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    errors = []
    for url in urls:
        try:
            urllib.request.urlretrieve(url, path)
            return
        except Exception as exc:
            errors.append(f'{url}: {exc}')
    raise RuntimeError(f'Could not download {path.name}:\n' + '\n'.join(errors))

def read_raw(path, columns):
    # One echocardiogram line starts with an extra comma, which shifts every value one column right.
    # Remove it so that line has the same 13 fields as the others (the raw file itself is left unchanged).
    lines = [line for line in path.read_text().splitlines() if line.strip()]
    fixed = [line[1:] if line.count(',') == len(columns) and line.startswith(',') else line for line in lines]
    for before, after in zip(lines, fixed):
        if before != after:
            print(f'  repaired shifted line: {before!r} -> {after!r}')
    return pd.read_csv(io.StringIO('\n'.join(fixed)), names=columns, na_values='?')

RAW, TARGETS = {}, {}
for name, (urls, columns, target) in SOURCES.items():
    path = DATASET_DIR / name / f'{name}.data'
    download(urls, path)
    RAW[name] = read_raw(path, columns)
    TARGETS[name] = target
    print(f'{name}: {path.relative_to(PROJECT_ROOT).as_posix()} ({path.stat().st_size:,} bytes), {RAW[name].shape}')

  repaired shifted line: ',?,?,77,?,?,?,?,?,2,?,name,2,?' -> '?,?,77,?,?,?,?,?,2,?,name,2,?'
echocardiogram: dataset/echocardiogram/echocardiogram.data (6,107 bytes), (132, 13)
iris: dataset/iris/iris.data (4,551 bytes), (150, 5)


## 4. Inspect before changing anything

Expected: Echocardiogram 132 rows × 13 columns with missing values (`?`), Iris 150 × 5 with no missing values.

In [5]:
EXPECTED_SHAPES = {'echocardiogram': (132, 13), 'iris': (150, 5)}

for name, data in RAW.items():
    target = TARGETS[name]
    print(f'=== {name} ===')
    print('Shape:', data.shape)
    assert data.shape == EXPECTED_SHAPES[name]
    print('Non-numeric columns:', list(data.select_dtypes(exclude='number').columns))
    print('Duplicate rows:', int(data.duplicated().sum()))
    missing = data.isna().sum()
    print('Missing values per column:', missing[missing > 0].to_dict())
    print('Target values (incl. missing):')
    print(data[target].value_counts(dropna=False).to_string())
    print()

=== echocardiogram ===
Shape: (132, 13)
Non-numeric columns: ['name']
Duplicate rows: 0
Missing values per column: {'survival': 2, 'still-alive': 1, 'age-at-heart-attack': 5, 'pericardial-effusion': 1, 'fractional-shortening': 8, 'epss': 15, 'lvdd': 11, 'wall-motion-score': 4, 'wall-motion-index': 1, 'mult': 4, 'group': 22, 'alive-at-1': 58}
Target values (incl. missing):
alive-at-1
NaN    58
0.0    50
1.0    24

=== iris ===
Shape: (150, 5)
Non-numeric columns: ['class']
Duplicate rows: 3
Missing values per column: {}
Target values (incl. missing):
class
Iris-setosa        50
Iris-versicolor    50
Iris-virginica     50



In [6]:
RAW['echocardiogram'].describe().T  # change to 'iris' to look at Iris

,count,mean,std,min,25%,50%,75%,max
survival,130.0,22.182923,15.858267,0.03,7.875,23.500,33.000,57.00
still-alive,131.0,0.328244,0.471377,0.00,0.000,0.000,1.000,1.00
age-at-heart-attack,127.0,62.925425,8.403757,35.00,57.000,62.000,68.000,86.00
pericardial-effusion,131.0,0.183206,0.388320,0.00,0.000,0.000,0.000,1.00
fractional-shortening,124.0,0.216734,0.107513,0.01,0.150,0.205,0.270,0.61
epss,117.0,12.164769,7.370159,0.00,7.000,11.000,16.100,40.00
lvdd,121.0,4.763157,0.810013,2.32,4.230,4.650,5.300,6.78
wall-motion-score,128.0,14.438125,5.018566,2.00,11.000,14.000,16.500,39.00
wall-motion-index,131.0,1.382748,0.453378,1.00,1.000,1.222,1.535,3.00
mult,128.0,0.776719,0.199078,0.14,0.713,0.786,0.857,2.00


## 5. Clean

**Echocardiogram**

- *Columns:* keep the 8 features the official GradTree code uses (`age-at-heart-attack`, `pericardial-effusion`, `fractional-shortening`, `epss`, `lvdd`, `wall-motion-score`, `wall-motion-index`, `mult`). `survival` and `still-alive` are dropped because the target is derived from them (leakage); `name` and `group` carry no information.
- *Target:* `alive-at-1` is missing for 58 of 132 patients. UCI says it is derived from `survival` and `still-alive`, but the labels in the file don't follow that description (label 1 goes with `still-alive = 1` and survival under 12 months, label 0 mostly with survival of 12+ months; the cross-table below shows this), so the missing labels can't be reliably rebuilt from those columns. As the paper did, they are filled with the most common label (0).
- *Missing feature values:* as the paper did, each column's missing values are filled with that column's most common value over the whole dataset, before splitting (official code: `data.fillna(data.mode().iloc[0])`).
- *Raw file:* line 50 of the UCI file starts with an extra comma, which section 3 removes. The copy in the GradTree repository has the same line fixed the same way, so we use the same data as the authors.
- *Class counts:* the paper's Table 16 lists 107 / 25, but running the official loader on the authors' own file gives 108 / 24, the same as here.

**Iris**: nothing to clean; all 4 numeric features are kept.

In [7]:
ECHO_FEATURES = ['age-at-heart-attack', 'pericardial-effusion', 'fractional-shortening', 'epss',
                 'lvdd', 'wall-motion-score', 'wall-motion-index', 'mult']

def clean_echocardiogram(data):
    target = data['alive-at-1']
    print('  alive-at-1 vs still-alive / survival >= 12 months (-1 = missing):')
    print(pd.crosstab(target.fillna(-1), [data['still-alive'].fillna(-1), data['survival'] >= 12]).to_string())
    print(f'  filling {int(target.isna().sum())} missing labels with the mode ({target.mode()[0]:g})')
    data = data.fillna(data.mode().iloc[0])  # mode of every column, incl. the label, over the whole dataset
    X = data[ECHO_FEATURES]
    y = (data['alive-at-1'] != 0).astype(int)  # official code: alive-at-1 != 0
    return X, y

def clean_iris(data):
    return data.drop(columns='class'), data['class']

CLEANERS = {'echocardiogram': clean_echocardiogram, 'iris': clean_iris}

DATASETS = {}
for name, data in RAW.items():
    print(f'=== {name} ===')
    X, y = CLEANERS[name](data)
    if DROP_DUPLICATES:
        keep = ~pd.concat([X, y], axis=1).duplicated()
        X, y = X[keep], y[keep]
    X, y = X.reset_index(drop=True), y.reset_index(drop=True)
    DATASETS[name] = (X, y)
    assert not X.isna().any().any() and not y.isna().any()
    print(f'  {data.shape[0]} -> {X.shape[0]} rows, {X.shape[1]} features, no missing values left')
    print('  class balance:', y.value_counts(normalize=True).round(3).to_dict(), '\n')

=== echocardiogram ===
  alive-at-1 vs still-alive / survival >= 12 months (-1 = missing):
still-alive  -1.0   0.0         1.0      
survival    False False True  False True 
alive-at-1                               
-1.0            1     2    41    11     3
 0.0            0     2    43     0     5
 1.0            0     0     0    24     0
  filling 58 missing labels with the mode (0)
  132 -> 132 rows, 8 features, no missing values left
  class balance: {0: 0.818, 1: 0.182} 

=== iris ===
  150 -> 150 rows, 4 features, no missing values left
  class balance: {'Iris-setosa': 0.333, 'Iris-versicolor': 0.333, 'Iris-virginica': 0.333} 



## 6. Shared pipeline: split → transform → SMOTE

The quantile transform is **fit on the training set only**, then applied to validation and test. Validation and test data are never resampled.

In [8]:
def finish_pipeline(X, y, seed):
    X = X.to_numpy(dtype=float)

    # Encode labels to 0..K-1 (Iris uses class names; PyTorch cross-entropy needs 0-based)
    le = LabelEncoder().fit(y)
    y = le.transform(y)

    # Paper: 80/20 train-test split, then 20% of the training data as validation (no stratification)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=seed)
    X_tr, X_valid, y_tr, y_valid = train_test_split(X_tr, y_tr, test_size=VAL_SIZE, random_state=seed)

    # Quantile transform: fit on (noisy copy of) train only, then transform the real values.
    # The noise breaks ties in features with many repeated values, e.g. the imputed modes.
    # Official code: noise and transform are seeded with 42 in every trial (only the split changes with the seed).
    X_fit = X_tr.copy()
    if QUANTILE_NOISE > 0:
        rng = np.random.RandomState(QT_SEED)
        stds = X_fit.std(axis=0, keepdims=True)
        X_fit += QUANTILE_NOISE / np.maximum(stds, QUANTILE_NOISE) * rng.randn(*X_fit.shape)
    qt = QuantileTransformer(n_quantiles=min(1000, len(X_tr)), output_distribution=QT_OUTPUT,
                             subsample=None, random_state=QT_SEED)
    qt.fit(X_fit)
    X_tr, X_valid, X_te = qt.transform(X_tr), qt.transform(X_valid), qt.transform(X_te)

    # SMOTE on the training set only, using the paper's rule
    n_train = len(y_tr)
    n_classes = len(np.unique(y_tr))
    minority_share = np.bincount(y_tr).min() / len(y_tr)
    smote_threshold = BALANCE_THRESHOLD / (n_classes - 1)
    smote_used = minority_share <= smote_threshold
    if smote_used:
        # Official code falls back to random oversampling when a class is too small for SMOTE's neighbours.
        # It doesn't seed SMOTE; we do, so the saved files are reproducible.
        try:
            X_tr, y_tr = SMOTE(random_state=seed).fit_resample(X_tr, y_tr)
        except ValueError:
            X_tr, y_tr = RandomOverSampler(random_state=seed).fit_resample(X_tr, y_tr)

    return dict(X_train=X_tr, X_valid=X_valid, X_test=X_te,
                y_train=y_tr, y_valid=y_valid, y_test=y_te,
                label_encoder=le, quantile_transformer=qt, train_rows_before_smote=n_train,
                minority_share=minority_share, smote_threshold=smote_threshold, smote_used=smote_used)

## 7. Save processed datasets and fitted transformers

Each dataset is saved to `preprocess/<name>_output/`, in the same format as the other datasets:

- `<name>_train.csv`, `<name>_valid.csv`, `<name>_test.csv`: transformed features, with the target (encoded to 0..K-1) as the last column.
- `<name>_preprocessor.joblib`: the fitted quantile transformer and label encoder. `inverse_transform` maps features and predictions back to the original values, which helps with error analysis.
- `<name>_preprocessing_summary.json`: row counts, missing values and SMOTE details, to check a run without opening the CSVs.

Only the `SEED` split (trial 0) is saved by default. Set `SAVE_ALL_SEEDS = True` to also write every seed to `seed_<seed>/` for 10-trial experiments.

In [9]:
def save_outputs(name, X, y, seed, out_dir):
    d = finish_pipeline(X, y, seed)
    out_dir.mkdir(parents=True, exist_ok=True)
    feature_names = list(X.columns)
    target = TARGETS[name]
    le = d['label_encoder']

    def save_split(part, features, labels):
        output = pd.DataFrame(features, columns=feature_names)
        output[target] = np.asarray(labels, dtype=np.int8)
        output.to_csv(out_dir / f'{name}_{part}.csv', index=False)

    save_split('train', d['X_train'], d['y_train'])
    save_split('valid', d['X_valid'], d['y_valid'])
    save_split('test', d['X_test'], d['y_test'])

    # No categorical features, so no target encoder; the keys stay so every preprocessor file has the same fields
    joblib.dump({
        'encoder': None,
        'quantile_transformer': d['quantile_transformer'],
        'label_encoder': le,
        'feature_names': feature_names,
        'categorical_features': [],
        'numeric_features': feature_names,
        'target': target,
        'random_seed': seed,
        'quantile_noise': QUANTILE_NOISE,
    }, out_dir / f'{name}_preprocessor.joblib')

    def counts(labels):
        return {int(label): int(count) for label, count in enumerate(np.bincount(labels))}

    summary = {
        'source_rows': int(len(RAW[name])),
        'rows_after_cleaning': int(len(X)),
        'feature_count': int(X.shape[1]),
        'features': feature_names,
        'target': target,
        'classes': [str(c) for c in le.classes_],  # original label of encoded class 0, 1, ...
        'original_target_counts': counts(le.transform(y)),
        'missing_counts': {column: int(count) for column, count in RAW[name][feature_names + [target]].isna().sum().items() if count},
        'imputation': (None if not RAW[name].isna().any().any()
                       else 'mode of full dataset, before split'),
        'split_rows': {'train': int(d['train_rows_before_smote']),
                       'valid': int(len(d['y_valid'])), 'test': int(len(d['y_test']))},
        'train_rows_after_smote': int(len(d['y_train'])),
        'minority_ratio_before_smote': float(d['minority_share']),
        'smote_threshold': float(d['smote_threshold']),
        'smote_applied': bool(d['smote_used']),
        'train_target_counts_after_smote': counts(d['y_train']),
        'random_seed': seed,
    }
    (out_dir / f'{name}_preprocessing_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

for name, (X, y) in DATASETS.items():
    out_dir = PREPROCESS_DIR / f'{name}_output'
    summary = save_outputs(name, X, y, SEED, out_dir)
    print(f'=== {name} ===')
    print(json.dumps(summary, indent=2))
    if SAVE_ALL_SEEDS:
        for seed in SEEDS:
            save_outputs(name, X, y, seed, out_dir / f'seed_{seed}')
        print(f'Also saved {len(SEEDS)} seeds to {out_dir.relative_to(PROJECT_ROOT).as_posix()}/seed_<seed>/')
    print('Saved files:')
    for path in sorted(out_dir.glob('*.*')):
        print(f'- {path.name} ({path.stat().st_size:,} bytes)')
    print()

=== echocardiogram ===
{
  "source_rows": 132,
  "rows_after_cleaning": 132,
  "feature_count": 8,
  "features": [
    "age-at-heart-attack",
    "pericardial-effusion",
    "fractional-shortening",
    "epss",
    "lvdd",
    "wall-motion-score",
    "wall-motion-index",
    "mult"
  ],
  "target": "alive-at-1",
  "classes": [
    "0",
    "1"
  ],
  "original_target_counts": {
    "0": 108,
    "1": 24
  },
  "missing_counts": {
    "age-at-heart-attack": 5,
    "pericardial-effusion": 1,
    "fractional-shortening": 8,
    "epss": 15,
    "lvdd": 11,
    "wall-motion-score": 4,
    "wall-motion-index": 1,
    "mult": 4,
    "alive-at-1": 58
  },
  "imputation": "mode of full dataset, before split",
  "split_rows": {
    "train": 84,
    "valid": 21,
    "test": 27
  },
  "train_rows_after_smote": 136,
  "minority_ratio_before_smote": 0.19047619047619047,
  "smote_threshold": 0.25,
  "smote_applied": true,
  "train_target_counts_after_smote": {
    "0": 68,
    "1": 68
  },
  "random

## 8. Sanity checks

1. No missing values may remain in any saved split.
2. Class proportions should be similar across train/valid/test (exactly equal only with stratified splits). Echocardiogram is small, so its test set has only ~20 rows and the shares move around.
3. Transformed training features should look roughly normal (mean ≈ 0, std ≈ 1 if `QT_OUTPUT='normal'`).
4. A quick CART should get above 90% accuracy on Iris. Echocardiogram is a hard, tiny dataset, so an untuned CART scores far lower there (in the paper, GradTree beats CART by a wide margin on Echocardiogram).

`load_split` reads the saved CSVs the way training code would, so these checks also test the saved files.

In [10]:
def load_split(name, out_dir=None):
    out_dir = out_dir or PREPROCESS_DIR / f'{name}_output'
    d = {}
    for part in ['train', 'valid', 'test']:
        frame = pd.read_csv(out_dir / f'{name}_{part}.csv')
        d[f'X_{part}'] = frame.iloc[:, :-1].to_numpy()
        d[f'y_{part}'] = frame.iloc[:, -1].to_numpy()
    return d

for name, (X, y) in DATASETS.items():
    d = load_split(name)
    print(f'=== {name} (saved split, seed {SEED}) ===')
    for part in ['train', 'valid', 'test']:
        assert np.isfinite(d[f'X_{part}']).all(), f'{part} still has missing or infinite values'
        labels = d[f'y_{part}']
        print(f'  {part:5s} rows {len(labels):4d} | class shares: {np.round(np.bincount(labels) / len(labels), 3)}')
    print(f'  train feature mean {d["X_train"].mean():.2f}, std {d["X_train"].std():.2f}')

    # CART over every seed (not saved), to check the pipeline isn't tuned to one split
    accs, f1s = [], []
    for seed in SEEDS:
        s = finish_pipeline(X, y, seed)
        pred = DecisionTreeClassifier(random_state=seed).fit(s['X_train'], s['y_train']).predict(s['X_test'])
        accs.append(accuracy_score(s['y_test'], pred))
        f1s.append(f1_score(s['y_test'], pred, average='macro'))
    print(f'  CART over {len(SEEDS)} seeds: accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}, '
          f'macro F1 {np.mean(f1s):.3f} ± {np.std(f1s):.3f}\n')

=== echocardiogram (saved split, seed 42) ===
  train rows  136 | class shares: [0.5 0.5]
  valid rows   21 | class shares: [0.857 0.143]
  test  rows   27 | class shares: [0.815 0.185]
  train feature mean 0.12, std 1.04


  CART over 10 seeds: accuracy 0.670 ± 0.067, macro F1 0.526 ± 0.081

=== iris (saved split, seed 42) ===
  train rows   96 | class shares: [0.333 0.312 0.354]
  valid rows   24 | class shares: [0.333 0.458 0.208]
  test  rows   30 | class shares: [0.333 0.3   0.367]
  train feature mean 0.03, std 1.22


  CART over 10 seeds: accuracy 0.940 ± 0.033, macro F1 0.937 ± 0.037



## Loading the splits later

In the GradTree or CART training code, every dataset loads the same way: features first, target last.
```python
import pandas as pd

train = pd.read_csv('preprocess/echocardiogram_output/echocardiogram_train.csv')
X_train, y_train = train.iloc[:, :-1].to_numpy(), train.iloc[:, -1].to_numpy()
```
Labels are already 0..K-1. `joblib.load('preprocess/iris_output/iris_preprocessor.joblib')['label_encoder']` maps Iris labels back to the class names.